# 🎙️ 코치 AI 목소리 데이터셋 자동 생성기
이 노트북은 긴 음성 파일(.m4a, .wav)을 AI가 학습하기 좋은 짧은 문장 단위로 자르고, 각 문장마다 인공지능(Whisper)이 받아쓰기를 해서 텍스트(metadata.csv)를 만들어주는 자동화 프로그램입니다.

## 1단계: 필수 프로그램 설치
아래 재생(▶️) 버튼을 눌러서 필요한 프로그램들을 설치하세요.

In [ ]:
!pip install -q faster-whisper pydub pandas
!apt-get install -y -q ffmpeg

## 2단계: 파일 업로드
왼쪽 폴더 아이콘(📁)을 누르고, 코치님의 녹음 파일(`.m4a`)들을 이곳에 드래그 앤 드롭해서 업로드해 주세요. (업로드가 완료될 때까지 기다려 주세요)

## 3단계: 자동 받아쓰기 및 오디오 자르기
업로드가 완료되면 아래 재생(▶️) 버튼을 눌러주세요. 인공지능이 파일을 분석해서 `wavs` 폴더에 잘게 자른 오디오 파일들을 만들고, `metadata.csv` 파일에 대본을 작성합니다.

In [ ]:
import os
import glob
import shutil
from pydub import AudioSegment
from faster_whisper import WhisperModel

os.makedirs('wavs', exist_ok=True)

# m4a 파일을 wav로 변환 (22050Hz, Mono)
m4a_files = glob.glob("*.m4a")
for f in m4a_files:
    print(f"{f} 변환 중...")
    audio = AudioSegment.from_file(f, format="m4a")
    audio = audio.set_frame_rate(22050).set_channels(1)
    audio.export(f.replace('.m4a', '.wav'), format="wav")
    
wav_files = glob.glob("*.wav")

print("\n🤖 인공지능 모델(Whisper)을 불러오는 중입니다... (약 1~2분 소요)")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")

metadata = []
segment_idx = 1

print("\n✂️ 오디오 분석 및 자르기 시작!")
for wav in wav_files:
    print(f"{wav} 처리 중...")
    segments, info = model.transcribe(wav, beam_size=5, language="ko")
    audio = AudioSegment.from_wav(wav)
    
    for segment in segments:
        text = segment.text.strip()
        if not text: continue
        
        # 0.2초 여유를 두고 자르기
        start_ms = max(0, int(segment.start * 1000) - 200)
        end_ms = min(len(audio), int(segment.end * 1000) + 200)
        
        chunk = audio[start_ms:end_ms]
        
        # 1.5초 미만 너무 짧은 소리는 학습에 방해되므로 제외
        if len(chunk) < 1500:
            continue
            
        out_name = f"segment_{segment_idx:04d}.wav"
        chunk.export(os.path.join("wavs", out_name), format="wav")
        
        # Piper 학습 포맷 (파일이름|대본)
        metadata.append(f"{out_name}|{text}")
        segment_idx += 1

with open("metadata.csv", "w", encoding="utf-8") as f:
    f.write("\n".join(metadata))

print(f"\n✅ 완료되었습니다! 총 {segment_idx-1}개의 문장 조각이 만들어졌습니다.")


## 4단계: 파일 압축해서 다운로드
완성된 파일들(`wavs` 폴더와 `metadata.csv`)을 하나의 압축 파일로 묶어서 컴퓨터로 다운로드합니다.

In [ ]:
import shutil
from google.colab import files

print("📦 압축 중입니다...")
shutil.make_archive("dataset", "zip", ".", "wavs")
shutil.move("dataset.zip", "/tmp/dataset_temp.zip")
shutil.make_archive("voice_dataset_final", "zip", ".", "metadata.csv")

# wavs 파일과 metadata.csv를 같이 압축
os.system('zip -r voice_dataset_final.zip wavs metadata.csv')

print("⬇️ 다운로드를 시작합니다...")
files.download('voice_dataset_final.zip')